# Phase 5 (LSI) — c-6 simple encoder-decoder: LORMA (fill the TODOs)

Multi-label statute identification: each case gets a **set** of statute IDs. Run the cells top to bottom.

- Cells marked **[ANCILLARY]** are plumbing: loading, tokenising, scoring, the writer, windowing, the loss, thresholds and the training loop. Nothing to change there, except `ROOT` in Cell A if your input path differs.
- **Cell E2 is yours**: every hyperparameter and design choice (`None`), the `LORMALinear` class and the `LSIHead`. It stops with a clear error at the first thing you haven't filled.


## Cell 0a — [ANCILLARY] imports + HF login

In [1]:
# --- Phase 1 c-6 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict


# HF login (Phase 1 cell 2), needed if the dataset download asks for auth
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## Cell 0b — [ANCILLARY] tokenizer (Phase 1)

In [2]:
# --- Phase 1 c-6 notebook, cell 8 (verbatim): SPECIAL_TOKENS, BPETokenizer, train_tokenizer ---


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok


## Cell 0c — [ANCILLARY] model classes (Phase 1)

In [3]:
# --- Phase 1 c-6 notebook, cells 20, 22 and 23 (verbatim, in this order) ---
# Cell 23 must come after cell 22: it redefines CausalSelfAttention (q/k/v_proj), which the
# saved decoder weights use. Without it, Cell A fails with qkv_proj key errors in decoder_layers.

# ---- cell 20 ----
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---- cell 22 ----
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Seq2SeqConfig:
    vocab_size: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    n_layers: int = 8

# ---------------------------------------------------------
# 1. Attention Modules
# ---------------------------------------------------------
class EncoderSelfAttention(nn.Module):
    """Bidirectional self-attention for the encoder (no causal mask)."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CausalSelfAttention(nn.Module):
    """Causal self-attention for the decoder (looks only at past tokens)."""
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.mask[:T, :T], float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CrossAttention(nn.Module):
    """Cross-attention: Query from Decoder, Key/Value from Encoder."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        B_c, T_c, C_c = context.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        kv = self.kv_proj(context).chunk(2, dim=-1)
        k, v = [t.view(B_c, T_c, self.n_heads, self.d_head).transpose(1, 2) for t in kv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

# ---------------------------------------------------------
# 2. Feed-Forward & Standard Blocks
# ---------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)


class EncoderBlock(nn.Module):
    """Standard bidirectional Transformer encoder block."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = EncoderSelfAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    """Standard causal Transformer decoder block with cross-attention."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout)
        
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = CrossAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), encoder_out)
        x = x + self.ffn(self.ln3(x))
        return x


# ---------------------------------------------------------
# 4. Hourglass Blocks (Encoder & Decoder versions)
# ---------------------------------------------------------
class EncoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([EncoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor):
        residual = x
        shortened_x = self.downsample(x) # No shift for Encoder
        for block in self.bottleneck:
            shortened_x = block(shortened_x)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual


class DecoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([DecoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor):
        residual = x
        x_shifted = shift_right(x, shift=self.factor - 1) # Shift required for Decoder causality
        shortened_x = self.downsample(x_shifted)
        for block in self.bottleneck:
            shortened_x = block(shortened_x, encoder_out)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual

# ---------------------------------------------------------
# 5. Top-Level Hourglass Seq2Seq Model
# ---------------------------------------------------------
class StandardEncoderDecoder(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)

        self.encoder_layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.encoder_ln = nn.LayerNorm(cfg.d_model)

        self.decoder_layers = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.decoder_ln = nn.LayerNorm(cfg.d_model)

        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def get_positional_embeddings(self, seq_len, device):
        return self.pos_emb(torch.arange(seq_len, device=device).unsqueeze(0))

    def encode(self, enc_idx):
        x = self.tok_emb(enc_idx) + self.get_positional_embeddings(enc_idx.size(1), enc_idx.device)
        for block in self.encoder_layers:
            x = block(x)
        return self.encoder_ln(x)

    def decode(self, dec_idx, encoder_out):
        x = self.tok_emb(dec_idx) + self.get_positional_embeddings(dec_idx.size(1), dec_idx.device)
        for block in self.decoder_layers:
            x = block(x, encoder_out)
        return self.lm_head(self.decoder_ln(x))

    def forward(self, enc_idx, dec_idx):
        return self.decode(dec_idx, self.encode(enc_idx))


# ---- cell 23 ----
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 560
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 2000
    max_seq_len: int = 256
    dropout: float = 0.15
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


@dataclass
class EncDecConfig(Seq2SeqConfig):
    enc_len: int = None      # tokens the encoder reads (prompt side)
    dec_len: int = None      # tokens the decoder learns to produce (continuation side)


class EncDecLM(nn.Module):
    def __init__(self, cfg: EncDecConfig):
        super().__init__()
        assert None not in (cfg.enc_len, cfg.dec_len), "set enc_len and dec_len in the config"
        self.cfg = cfg
        self.core = StandardEncoderDecoder(cfg)

    def encode(self, enc_idx):
        return self.core.encode(enc_idx)

    def decode(self, dec_idx, encoder_out):
        return self.core.decode(dec_idx, encoder_out)

    def forward(self, enc_idx, dec_idx):
        return self.core(enc_idx, dec_idx)


## Cell A — [ANCILLARY] load c-6 + tokenizer

In [4]:
# =====================================================================================
# CELL A — load the Phase 1 c-6 encoder-decoder and its tokenizer
# Expect <All keys matched successfully>, 55.13M params, vocab 11022, d_model 560, 8 layers, max_seq_len 1536
# =====================================================================================
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudraalt/phase-1-c-6-simple-encoder-decoder"   # TODO: the folder the find command showed (up to, not including, cs781_phase1_full)
device = torch.device("cuda")

tok = pickle.load(open(f"{ROOT}/cs781_phase1_full/model_encdec/bpe_tokenizer.pkl", "rb"))   # c-6's own tokenizer
ck = torch.load(f"{ROOT}/cs781_phase1_full/model_encdec/model_final.pt", map_location="cpu", weights_only=False)
cfg = EncDecConfig(**ck["config"])
lm = EncDecLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| n_layers", cfg.n_layers, "| max_seq_len", cfg.max_seq_len)


<All keys matched successfully>
params 55.13008 M | vocab 11022 | d_model 560 | n_layers 8 | max_seq_len 1536


## Cell B — [ANCILLARY] encoder features check

In [5]:
# =====================================================================================
# CELL B — per-token vectors from the c-6 ENCODER + sanity check
# The encoder is bidirectional: every token sees the tokens on both sides. The decoder is not used.
# =====================================================================================
def features(idx):                        # idx: (B, T) LongTensor, T <= 1536
    c = lm.core
    T = idx.size(1)
    assert T <= c.cfg.max_seq_len         # learned position table has 1536 rows
    x = c.tok_emb(idx) + c.get_positional_embeddings(T, idx.device)
    for blk in c.encoder_layers:          # 8 bidirectional EncoderBlocks
        x = blk(x)
    return c.encoder_ln(x)                # (B, T, 560)

# Sanity check: same as the model's own encoder
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    f, ref = features(idx), lm.encode(idx)
print("max diff", (f - ref).abs().max().item(), tuple(f.shape))   # expect 0.0, (1, 64, 560)


max diff 0.0 (1, 64, 560)


## Cell C — [ANCILLARY] LSI data + statute set

In [6]:
# =====================================================================================
# CELL C — [ANCILLARY] Phase 5 (LSI) data: load train / dev / test cases and the statute set
# One case = one long judgment -> a SET of statute IDs (multi-label).
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json
import numpy as np
from collections import Counter
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "Exploration-Lab/CS-781-Capstone"
P5_DIR = "Phase_5_LSI"
print("Phase 5 files on HF:", [f for f in list_repo_files(REPO, repo_type="dataset") if "Phase_5" in f])

def get5(name):
    """Reads Phase_5_LSI/<name>. Accepts plain JSON or JSON-lines."""
    p = hf_hub_download(REPO, f"{P5_DIR}/{name}", repo_type="dataset")
    with open(p, encoding="utf-8") as f:
        txt = f.read()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

_raw = get5("Train/train.json")
_first = _raw[0] if isinstance(_raw, list) else next(iter(_raw.values()))
print("train layout:", type(_raw).__name__, len(_raw), "| first record keys:",
      list(_first) if isinstance(_first, dict) else type(_first).__name__)

TEXT_FIELDS  = ("text", "document", "case_text", "judgment", "sentences")
LABEL_FIELDS = ("labels", "label", "statutes", "statute_ids", "target", "y")

def _as_label_list(lab, k):
    """Any of [1, 7], "1,7", 7, None -> sorted list of unique ints."""
    if lab is None:
        return None
    if isinstance(lab, str):
        lab = [x for x in lab.replace(";", ",").split(",") if x.strip()]
    if not isinstance(lab, (list, tuple)):
        lab = [lab]
    out = sorted({int(x) for x in lab})
    return out

def as_cases(raw, name):
    """-> {case_id: {"text": str, "labels": [int, ...] or None}}, in file order.
    Accepts a list of dicts, {case_id: {...}}, a one-key wrapper like {"data": [...]},
    or columns {"id": [...], "text": [...], ("labels": [...])}.
    Repeated ids: exact duplicates dropped; differing repeats kept as "<id>#dup<n>" (train/dev),
    dropped with a warning on test (one prediction per id)."""
    while isinstance(raw, dict) and len(raw) == 1 and isinstance(next(iter(raw.values())), (list, dict)):
        raw = next(iter(raw.values()))
    if isinstance(raw, dict) and "id" in raw and isinstance(raw["id"], list):
        keys = list(raw)
        raw = [{k: raw[k][i] for k in keys} for i in range(len(raw["id"]))]
    items = [(d.get("id", n), d) for n, d in enumerate(raw)] if isinstance(raw, list) else list(raw.items())
    cases, n_exact, n_diff = {}, 0, 0
    for k, d in items:
        assert isinstance(d, dict), f"case {k}: unexpected entry {type(d).__name__}"
        k = str(d.get("id", k))
        tf = next((f for f in TEXT_FIELDS if f in d), None)
        assert tf is not None, f"case {k}: no text field (keys {list(d)})"
        text = d[tf]
        if isinstance(text, list):                    # a list of sentences/paragraphs
            text = " ".join(map(str, text))
        lf = next((f for f in LABEL_FIELDS if f in d), None)
        rec = {"text": str(text), "labels": _as_label_list(d[lf] if lf else None, k)}
        if k in cases:
            if cases[k] == rec:
                n_exact += 1
                continue
            if name == "test":
                print(f"WARNING test id {k} appears twice with different text: keeping the first")
                continue
            n_diff += 1
            j = 1
            while f"{k}#dup{j}" in cases:
                j += 1
            k = f"{k}#dup{j}"
        cases[k] = rec
    print(f"{name}: {len(items)} records -> {len(cases)} cases | exact duplicates dropped {n_exact} | "
          f"repeated ids with different text/labels kept {n_diff}")
    return cases

train = as_cases(_raw, "train")
dev   = as_cases(get5("Train/dev.json"), "dev")
test  = as_cases(get5("Test/test.json"), "test")
del _raw

assert all(d["labels"] for s in (train, dev) for d in s.values()), \
    "every train/dev case needs at least one statute: check LABEL_FIELDS"
_tl = sum(d["labels"] is not None for d in test.values())
if _tl:
    print(f"test has a label field on {_tl} cases (ignored, set to None)")
for d in test.values():
    d["labels"] = None

# -------------------------------------------------------------------------------------
# Statute set = every ID seen in train or dev. Model output j <-> statute STATUTES[j].
# -------------------------------------------------------------------------------------
STATUTES = sorted({l for s in (train, dev) for d in s.values() for l in d["labels"]})
S2J = {s: j for j, s in enumerate(STATUTES)}
N_LABELS = len(STATUTES)
VALID_STATUTES = set(STATUTES)

tr_cnt = Counter(l for d in train.values() for l in d["labels"])
dv_cnt = Counter(l for d in dev.values() for l in d["labels"])
print(f"\n{N_LABELS} statutes | ID range {STATUTES[0]}..{STATUTES[-1]} | "
      f"in train {len(tr_cnt)} | in dev {len(dv_cnt)} | dev-only (never in train) {sorted(set(dv_cnt) - set(tr_cnt))}")
for name, split in (("train", train), ("dev", dev)):
    n = np.array([len(d["labels"]) for d in split.values()])
    print(f"{name:5s}: {len(split):6d} cases | statutes per case min {n.min()} mean {n.mean():.2f} "
          f"median {np.median(n):.0f} max {n.max()}")
print(f"test : {len(test):6d} cases")

f = np.array([tr_cnt.get(s, 0) for s in STATUTES])
for lo, hi in ((0, 0), (1, 5), (6, 20), (21, 100), (101, 10**9)):
    print(f"  statutes with {lo}-{hi if hi < 10**9 else 'inf'} train cases: {int(((f >= lo) & (f <= hi)).sum())}")
print("most frequent:", tr_cnt.most_common(10))
print("rarest       :", sorted(tr_cnt.items(), key=lambda x: x[1])[:10])
print("first test ids:", list(test)[:3])
print(f"train∩dev ids {len(set(train) & set(dev))} | train∩test {len(set(train) & set(test))}")


Phase 5 files on HF: ['Phase_5_LSI/Test/test.json', 'Phase_5_LSI/Train/dev.json', 'Phase_5_LSI/Train/train.json']


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

train layout: list 42750 | first record keys: ['id', 'text', 'labels']
train: 42750 records -> 42750 cases | exact duplicates dropped 0 | repeated ids with different text/labels kept 0


Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

dev: 10181 records -> 10181 cases | exact duplicates dropped 0 | repeated ids with different text/labels kept 0


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

test: 13019 records -> 13019 cases | exact duplicates dropped 0 | repeated ids with different text/labels kept 0

100 statutes | ID range 0..99 | in train 100 | in dev 100 | dev-only (never in train) []
train:  42750 cases | statutes per case min 1 mean 3.70 median 3 max 31
dev  :  10181 cases | statutes per case min 1 mean 3.92 median 3 max 28
test :  13019 cases
  statutes with 0-0 train cases: 0
  statutes with 1-5 train cases: 0
  statutes with 6-20 train cases: 0
  statutes with 21-100 train cases: 0
  statutes with 101-inf train cases: 100
most frequent: [(5, 11730), (34, 10380), (97, 7880), (43, 7728), (39, 7120), (12, 4324), (13, 4136), (10, 4134), (77, 4046), (94, 3835)]
rarest       : [(84, 272), (25, 287), (23, 304), (92, 309), (66, 309), (59, 310), (50, 318), (74, 319), (27, 329), (15, 336)]
first test ids: ['100002997', '100004477', '1000086']
train∩dev ids 0 | train∩test 0


## Cell D — [ANCILLARY] tokenise, multi-label scorer, writer, baseline

In [7]:
# =====================================================================================
# CELL D — [ANCILLARY] document tokenisation, multi-label scorer, prediction writer, baseline
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json, os, zipfile
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

PAD_ID = SPECIAL_TOKENS.index("<pad>")   # 0
UNK_ID = SPECIAL_TOKENS.index("<unk>")   # 1

def encode_doc(text):
    ids = tok.encode(text)
    return ids if ids else [UNK_ID]

# Tokenise every case once: d["ids"] = token ids of the whole judgment (re-run if you change encode_doc)
for name, split in (("train", train), ("dev", dev), ("test", test)):
    for d in tqdm(split.values(), desc=f"tokenising {name}"):
        d["ids"] = encode_doc(d["text"])

# Length statistics in c-6 tokens, to choose WINDOW (<= 1536), STRIDE and MAX_WINDOWS in Cell E2
for name, split in (("train", train), ("dev", dev), ("test", test)):
    L = np.array([len(d["ids"]) for d in split.values()])
    print(f"{name:5s} tokens: min {L.min()} median {np.median(L):.0f} p90 {np.percentile(L, 90):.0f} "
          f"p99 {np.percentile(L, 99):.0f} max {L.max()} | > 1536: {(L > 1536).mean():.1%}")


# -------------------------------------------------------------------------------------
# Local scorer. pred / gold are SETS of statute IDs per case.
#   macro-F1 (primary): mean per-statute F1 over the statutes in the REFERENCE (gold) labels of the
#                       scored split, the way the spec says the evaluator sets the label dimension.
#                       A statute that is never gold but is predicted only costs micro precision.
#   macro-F1 (all):     the same mean over every statute in STATUTES (stricter; shown for reference).
#   micro-F1, subset accuracy (exact set match): secondary metrics.
# -------------------------------------------------------------------------------------
def ml_scores(gold_sets, pred_sets):
    tp, fp, fn = Counter(), Counter(), Counter()
    exact = 0
    for g, p in zip(gold_sets, pred_sets):
        g, p = set(g), set(p)
        exact += g == p
        for s in g & p: tp[s] += 1
        for s in p - g: fp[s] += 1
        for s in g - p: fn[s] += 1
    def f1(s):
        P = tp[s] / (tp[s] + fp[s]) if tp[s] + fp[s] else 0.0
        R = tp[s] / (tp[s] + fn[s]) if tp[s] + fn[s] else 0.0
        return P, R, (2 * P * R / (P + R) if P + R else 0.0), tp[s] + fn[s]
    ref = sorted({s for g in gold_sets for s in g})
    per = {s: f1(s) for s in sorted(set(ref) | set(STATUTES))}
    TP, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    mP = TP / (TP + FP) if TP + FP else 0.0
    mR = TP / (TP + FN) if TP + FN else 0.0
    return {"f1": float(np.mean([per[s][2] for s in ref])),
            "f1_all": float(np.mean([per[s][2] for s in STATUTES])),
            "p": float(np.mean([per[s][0] for s in ref])), "r": float(np.mean([per[s][1] for s in ref])),
            "micro_f1": 2 * mP * mR / (mP + mR) if mP + mR else 0.0, "micro_p": mP, "micro_r": mR,
            "subset_acc": exact / max(1, len(gold_sets)), "per": per}

def evaluate(pred, docs, show=True):
    """pred: {case_id: [statute IDs]}. Scores every case in docs."""
    ids = list(docs)
    res = ml_scores([docs[k]["labels"] for k in ids], [pred[k] for k in ids])
    if show:
        n_pred = np.mean([len(pred[k]) for k in ids])
        print(f"macro-F1 {res['f1']:.4f} (P {res['p']:.4f} R {res['r']:.4f}) | macro-F1 over all statutes {res['f1_all']:.4f} | "
              f"micro-F1 {res['micro_f1']:.4f} (P {res['micro_p']:.4f} R {res['micro_r']:.4f}) | "
              f"subset acc {res['subset_acc']:.4f} | statutes predicted per case {n_pred:.2f}")
    return res

def per_statute_report(res, k=15):
    """Per-statute F1 with train frequency: the rarest and the worst statutes (for the report)."""
    rows = [(s, *res["per"][s], tr_cnt.get(s, 0)) for s in STATUTES]
    print(f"{'statute':>8s} {'P':>6s} {'R':>6s} {'F1':>6s} {'dev n':>6s} {'train n':>8s}")
    for title, key in (("rarest in train", lambda r: r[5]), ("lowest F1 (dev n > 0)", lambda r: (r[3], -r[4]))):
        print(f"-- {title}")
        for s, P, R, F, n, ntr in sorted([r for r in rows if r[4] > 0], key=key)[:k]:
            print(f"{s:8d} {P:6.3f} {R:6.3f} {F:6.3f} {n:6d} {ntr:8d}")
    for lo, hi in ((1, 5), (6, 20), (21, 100), (101, 10**9)):
        f = [r[3] for r in rows if r[4] > 0 and lo <= r[5] <= hi]
        if f:
            print(f"statutes with {lo}-{hi if hi < 10**9 else 'inf'} train cases: {len(f)} | mean dev F1 {np.mean(f):.4f}")

try:   # cross-check the macro / micro numbers against sklearn once
    from sklearn.metrics import f1_score
    from sklearn.preprocessing import MultiLabelBinarizer
    _g, _p = [[STATUTES[0]], [STATUTES[0], STATUTES[-1]], [STATUTES[-1]]], [[STATUTES[0]], [STATUTES[0]], []]
    _m = MultiLabelBinarizer(classes=sorted({s for g in _g for s in g})).fit(_g)
    _r = ml_scores(_g, _p)
    assert abs(_r["f1"] - f1_score(_m.transform(_g), _m.transform(_p), average="macro", zero_division=0)) < 1e-9
    assert abs(_r["micro_f1"] - f1_score(_m.transform(_g), _m.transform(_p), average="micro", zero_division=0)) < 1e-9
    print("scorer matches sklearn macro / micro F1")
except ImportError:
    pass


# -------------------------------------------------------------------------------------
# Writer: prediction.json = LIST of {"id", "pred": [int, ...]} in test order, zipped.
# Checks: one entry per test id, plain Python ints, no duplicates, every ID a known statute.
# -------------------------------------------------------------------------------------
def write_prediction(pred, test_docs, out_dir="/kaggle/working"):
    os.makedirs(out_dir, exist_ok=True)
    assert set(pred) >= set(test_docs), "missing test ids"
    out, n_empty = [], 0
    for case_id in test_docs:
        p = sorted({int(s) for s in pred[case_id]})      # numpy ints -> plain int, deduplicated
        bad = [s for s in p if s not in VALID_STATUTES]
        assert not bad, f"{case_id}: unknown statute IDs {bad}"
        n_empty += not p
        out.append({"id": case_id, "pred": p})
    assert len(out) == len(test_docs) and len({o["id"] for o in out}) == len(out), "one entry per test case"
    if n_empty:
        print(f"WARNING {n_empty} test cases have an empty pred list (the spec does not say if [] is accepted)")
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)
    back = json.load(open(path))
    assert isinstance(back, list) and all(set(o) == {"id", "pred"} and isinstance(o["pred"], list)
                                          and all(type(s) is int for s in o["pred"])
                                          and len(set(o["pred"])) == len(o["pred"]) for o in back)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
    n = [len(o["pred"]) for o in out]
    print(f"{len(out)} cases | statutes per case mean {np.mean(n):.2f} max {max(n)} | "
          f"distinct statutes predicted {len({s for o in out for s in o['pred']})} -> {zpath}")


# -------------------------------------------------------------------------------------
# Baseline: the K most frequent train statutes for every case (K = median statutes per train case).
# Checks the scorer and writer end to end before any modelling.
# -------------------------------------------------------------------------------------
_K = int(np.median([len(d["labels"]) for d in train.values()]))
_top = [s for s, _ in tr_cnt.most_common(_K)]
print(f"baseline: top-{_K} statutes {_top} for every case")
evaluate({k: _top for k in dev}, dev)
write_prediction({k: _top for k in test}, test, out_dir="/kaggle/working/baseline")   # format check


tokenising train:   0%|          | 0/42750 [00:00<?, ?it/s]

tokenising dev:   0%|          | 0/10181 [00:00<?, ?it/s]

tokenising test:   0%|          | 0/13019 [00:00<?, ?it/s]

train tokens: min 2 median 717 p90 4626 p99 14020 max 158155 | > 1536: 33.0%
dev   tokens: min 4 median 911 p90 5031 p99 15324 max 129542 | > 1536: 38.1%
test  tokens: min 2 median 899 p90 4790 p99 14810 max 256349 | > 1536: 36.9%
scorer matches sklearn macro / micro F1
baseline: top-3 statutes [5, 34, 97] for every case
macro-F1 0.0118 (P 0.0074 R 0.0300) | macro-F1 over all statutes 0.0118 | micro-F1 0.2131 (P 0.2456 R 0.1882) | subset acc 0.0002 | statutes predicted per case 3.00
13019 cases | statutes per case mean 3.00 max 3 | distinct statutes predicted 3 -> /kaggle/working/baseline/submission.zip


## Cell E0 — [ANCILLARY] windows, loss, thresholds, training loop

In [8]:
# =====================================================================================
# CELL E0 — [ANCILLARY] shared plumbing for the c-6 Phase 5 (LSI) experiments, Phase 2/4 style
# Run once after Cells 0, A, B, C, D. Then run Cell E2 (LORMA).
# Needs from earlier cells: lm, cfg, device, features, train, dev, test (with d["ids"]),
#   STATUTES, S2J, N_LABELS, tr_cnt, evaluate, write_prediction
#
# Windows exactly as in Phase 4: full-length windows of WINDOW tokens every STRIDE tokens,
# the last one ending at the document end; a short case is one window of its own length.
# The encoder has no padding mask, so windows are never padded: batches hold windows of
# IDENTICAL length. Every window of a case is trained with that case's full statute set
# (multi-hot target), and at prediction time the case's window probabilities are combined
# into one probability per statute, then thresholded into a set of statute IDs.
# Nothing in this cell is a model design or a hyperparameter choice: every choice
# (window, aggregation, threshold, weighting) is an argument you set in Cell E2.
# =====================================================================================
import math, time, random
import numpy as np
from collections import defaultdict, Counter
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

FEAT_DIM = cfg.d_model   # 560 for c-6

# The decoder is never used for classification; dropping it frees GPU memory. tok_emb stays (shared with lm_head).
lm.core.decoder_layers = None
lm.core.decoder_ln = None
torch.cuda.empty_cache()


# -------------------------------------------------------------------------------------
# c-6 encoder path (identical to Cell B features(), but on any backbone object)
# -------------------------------------------------------------------------------------
# FAST_ATTENTION: same maths as EncoderSelfAttention.forward (same weights, LORMA wrappers
# included), but softmax(QK^T/sqrt(d))V runs in PyTorch's fused scaled_dot_product_attention
# instead of building the full (T x T) score matrix by hand. Checked and timed at the end of
# this cell; it is switched off automatically if it is not faster or not equal.
FAST_ATTENTION = True

def _self_attn_fast(attn, x):
    B, T, C = x.shape
    qkv = attn.qkv_proj(x).chunk(3, dim=-1)
    q, k, v = [t.view(B, T, attn.n_heads, attn.d_head).transpose(1, 2) for t in qkv]
    out = F.scaled_dot_product_attention(q, k, v, dropout_p=attn.attn_dropout.p if attn.training else 0.0)
    out = out.transpose(1, 2).contiguous().view(B, T, C)
    return attn.resid_dropout(attn.out_proj(out))


# GRAD_CHECKPOINT: memory saver for long windows, same maths. During training each block's
# inner activations (incl. the (B, heads, T, T) attention matrix) are not kept for backward;
# the block is recomputed in the backward pass instead (~30% slower, far less GPU memory).
# d_head = 70 is not a multiple of 8, so PyTorch's memory-efficient / flash attention kernels
# can't be used and SDPA falls back to the plain kernel, which stores that T x T matrix.
GRAD_CHECKPOINT = False
from torch.utils.checkpoint import checkpoint

def _block(blk, x):
    if FAST_ATTENTION:                          # EncoderBlock.forward with the fused attention
        x = x + _self_attn_fast(blk.self_attn, blk.ln1(x))
        return x + blk.ffn(blk.ln2(x))
    return blk(x)


def encoder_forward(backbone, idx):
    c = backbone.core
    assert idx.size(1) <= c.cfg.max_seq_len, "window longer than 1536"
    x = c.tok_emb(idx) + c.get_positional_embeddings(idx.size(1), idx.device)
    use_ckpt = GRAD_CHECKPOINT and torch.is_grad_enabled() and backbone.training
    for blk in c.encoder_layers:
        x = checkpoint(_block, blk, x, use_reentrant=False) if use_ckpt else _block(blk, x)
    return c.encoder_ln(x)                      # (B, T, 560)


def encoder_linears(backbone, which):
    """Yield (parent, attr, nn.Linear) for the chosen linears in every encoder block.
    which ⊆ {"qkv_proj", "out_proj", "ffn_in", "ffn_out"}.
    c-6 encoder attention has ONE fused qkv_proj (560 -> 1680), not q/k/v_proj."""
    for blk in backbone.core.encoder_layers:
        for w in which:
            if w in ("qkv_proj", "out_proj"):
                parent, attr = blk.self_attn, w
            elif w == "ffn_in":
                parent, attr = blk.ffn.net, "0"     # 560 -> d_ff
            elif w == "ffn_out":
                parent, attr = blk.ffn.net, "2"     # d_ff -> 560
            else:
                raise ValueError(w)
            yield parent, attr, getattr(parent, attr)


def wrap_linears(backbone, which, make_wrapper):
    """Replace each chosen linear by make_wrapper(linear). Returns how many were wrapped."""
    todo = list(encoder_linears(backbone, which))
    for parent, attr, lin in todo:
        assert isinstance(lin, nn.Linear), f"{attr} is already wrapped (re-run Cell A to start fresh)"
        setattr(parent, attr, make_wrapper(lin))
    return len(todo)


class LSIModel(nn.Module):
    """backbone (c-6 encoder path) -> head -> N_LABELS logits per WINDOW (one per statute).
    head: YOUR module, (B, T, 560) token vectors of B windows -> (B, N_LABELS) logits."""
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone, self.head = backbone, head
    def forward(self, idx):
        return self.head(encoder_forward(self.backbone, idx))   # (B, N_LABELS)


def identity_probe():
    """64 real tokens from a training case, for the 'adapter = identity at init' check."""
    d = next(iter(train.values()))
    return torch.tensor([d["ids"][:64]], device=device)


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


# -------------------------------------------------------------------------------------
# Windows (Phase 2 make_windows, unchanged) + which windows stand for the case
#   DOC_REPR "all" = every window (exactly Phase 2). Judgments are long, so you can also keep
#   at most max_windows of them: "first", "last", "first_last" (half from each end) or
#   "spread" (evenly over the whole case). Windows stay in document order.
# -------------------------------------------------------------------------------------
DOC_REPRS = ("all", "first", "last", "first_last", "spread")

def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]


def select_windows(wins, doc_repr, max_windows):
    n = len(wins)
    if doc_repr == "all" or n <= max_windows:
        return wins
    if doc_repr == "first":
        return wins[:max_windows]
    if doc_repr == "last":
        return wins[-max_windows:]
    if doc_repr == "first_last":
        h = (max_windows + 1) // 2
        return wins[:h] + wins[n - (max_windows - h):] if max_windows > h else wins[:h]
    if doc_repr == "spread":
        idx = sorted({round(i * (n - 1) / max(1, max_windows - 1)) for i in range(max_windows)})
        return [wins[i] for i in idx]
    raise ValueError(doc_repr)


def build_examples(docs, window, stride, doc_repr="all", max_windows=None):
    """One example per window: {"doc", "w", "ids", "labels"} (labels = the case's statute set, None for test)."""
    assert window <= cfg.max_seq_len and doc_repr in DOC_REPRS
    assert doc_repr == "all" or max_windows, "set max_windows for this doc_repr"
    examples = []
    for doc_id, d in tqdm(docs.items(), desc="windowing", leave=False):
        ids = d["ids"]
        wins = select_windows(make_windows(len(ids), window, stride), doc_repr, max_windows)
        for w, (s, e) in enumerate(wins):
            examples.append({"doc": doc_id, "w": w, "ids": ids[s:e], "labels": d["labels"]})
    per = Counter(ex["doc"] for ex in examples)
    print(f"{len(per)} cases -> {len(examples)} windows | windows per case: mean {len(examples)/len(per):.2f} max {max(per.values())}")
    return examples


def make_batches(examples, batch_size, shuffle):
    """Batches of windows with IDENTICAL length (no padding needed)."""
    by_len = defaultdict(list)
    for ex in examples:
        by_len[len(ex["ids"])].append(ex)
    out = []
    for exs in by_len.values():
        if shuffle:
            random.shuffle(exs)
        out += [exs[i:i + batch_size] for i in range(0, len(exs), batch_size)]
    if shuffle:
        random.shuffle(out)
    return out


def multi_hot(statutes):
    v = torch.zeros(N_LABELS)
    v[[S2J[s] for s in statutes]] = 1.0
    return v


def to_tensors(batch):
    b = {"ids": torch.tensor([ex["ids"] for ex in batch], device=device)}
    if batch[0]["labels"] is not None:
        b["labels"] = torch.stack([multi_hot(ex["labels"]) for ex in batch]).to(device)   # (B, N_LABELS) 0/1
    return b


# -------------------------------------------------------------------------------------
# Loss: one independent sigmoid per statute (binary relevance), BCE against the multi-hot target.
#   pos_weight (N_LABELS,): multiplies the loss on POSITIVE targets of each statute, the usual
#   PyTorch way to push recall up on rare statutes. make_pos_weight builds it from train counts:
#     "none"         -> no weighting
#     "neg_pos"      -> (#cases without the statute) / (#cases with it)
#     "sqrt_neg_pos" -> square root of that (milder)
#   cap: optional upper limit on any weight (very rare statutes get huge ratios). Your choice.
# -------------------------------------------------------------------------------------
def bce_loss(logits, targets, pos_weight=None):
    return F.binary_cross_entropy_with_logits(logits.float(), targets, pos_weight=pos_weight)


def make_pos_weight(scheme, cap=None):
    assert scheme in ("none", "neg_pos", "sqrt_neg_pos"), "set POS_WEIGHT_SCHEME"
    if scheme == "none":
        return None
    n = len(train)
    pos = torch.tensor([max(1, tr_cnt.get(s, 0)) for s in STATUTES], dtype=torch.float)
    w = (n - pos) / pos
    if scheme == "sqrt_neg_pos":
        w = w.sqrt()
    if cap is not None:
        w = w.clamp(max=cap)
    print(f"pos_weight: min {w.min():.2f} median {w.median():.2f} max {w.max():.2f}")
    return w.to(device)


# -------------------------------------------------------------------------------------
# Prediction: per-window sigmoid probabilities -> one probability vector per case -> statute set
#   agg (yours): "mean" = average over the case's windows, "max" = largest window probability
#   threshold (yours): one float for every statute (global) or an array of N_LABELS (per statute)
#   min_labels (yours): if fewer statutes pass the threshold, take the min_labels most probable
#                       ones instead (0 = allow an empty set; the spec does not say if [] is accepted)
# -------------------------------------------------------------------------------------
AGGS = ("mean", "max")

@torch.no_grad()
def predict_probs(model, examples, batch_size, agg):
    """-> {case_id: np.array (N_LABELS,)} aggregated window probabilities."""
    assert agg in AGGS, agg
    model.eval()
    per_doc = defaultdict(list)
    for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc="predict", leave=False):
        with torch.autocast("cuda", dtype=torch.float16):
            pr = model(to_tensors(batch)["ids"]).float().sigmoid().cpu().numpy()
        for ex, p in zip(batch, pr):
            per_doc[ex["doc"]].append(p)
    if agg == "mean":
        return {d: np.mean(v, axis=0) for d, v in per_doc.items()}
    return {d: np.max(v, axis=0) for d, v in per_doc.items()}


def decide(probs, threshold, min_labels=0):
    """probs {case_id: (N_LABELS,)} -> {case_id: [statute IDs]}."""
    thr = np.broadcast_to(np.asarray(threshold, dtype=float), (N_LABELS,))
    out = {}
    for d, p in probs.items():
        js = np.flatnonzero(p >= thr)
        if len(js) < min_labels:
            js = np.argsort(-p)[:min_labels]
        out[d] = [STATUTES[j] for j in js]
    return out


def predict(model, examples, batch_size, agg, threshold, min_labels=0):
    return decide(predict_probs(model, examples, batch_size, agg), threshold, min_labels)


# -------------------------------------------------------------------------------------
# Threshold selection on dev (the spec allows dev for this). grid = the candidate values (yours).
#   "global":    one threshold for every statute, the grid value with the best dev macro-F1
#   "per_label": start from the best global value, then for each statute separately the grid
#                value with the best F1 on that statute (statutes with no dev positives keep the
#                global value). Per-label thresholds fit dev closely: rare statutes have very few
#                dev cases, so expect dev macro-F1 to overstate the test score.
# -------------------------------------------------------------------------------------
def tune_thresholds(probs, docs, mode, grid, min_labels=0):
    assert mode in ("global", "per_label") and grid is not None and len(grid), "set THRESH_MODE / THRESH_GRID"
    ids = list(docs)
    P = np.stack([probs[k] for k in ids])                                   # (n_cases, N_LABELS)
    Y = np.stack([multi_hot(docs[k]["labels"]).numpy() for k in ids]).astype(bool)
    scores = [(evaluate(decide(probs, t, min_labels), docs, show=False)["f1"], t) for t in grid]
    best_f1, best_t = max(scores)
    print("global:", " ".join(f"{t:.2f}->{f:.4f}" for f, t in scores), f"| best {best_t} (dev macro-F1 {best_f1:.4f})")
    if mode == "global":
        return float(best_t)
    thr = np.full(N_LABELS, float(best_t))
    for j in range(N_LABELS):
        if not Y[:, j].any():
            continue
        best = (-1.0, best_t)
        for t in grid:
            pj = P[:, j] >= t
            tp = (pj & Y[:, j]).sum(); fp = (pj & ~Y[:, j]).sum(); fn = (~pj & Y[:, j]).sum()
            f = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
            best = max(best, (f, t))
        thr[j] = best[1]
    f = evaluate(decide(probs, thr, min_labels), docs, show=False)["f1"]
    print(f"per_label: dev macro-F1 {f:.4f} | thresholds min {thr.min():.2f} median {np.median(thr):.2f} max {thr.max():.2f}")
    return thr


# -------------------------------------------------------------------------------------
# Training loop (Phase 4 train_model). loss_fn(logits, batch_tensors) comes from the technique
# cell. Keeps the epoch with the best dev macro-F1 (case level, fixed threshold / min_labels).
# lr_lambda: optional step -> multiplier (your schedule), or None for constant LR.
# -------------------------------------------------------------------------------------
def train_model(model, train_ex, val_ex, val_docs, loss_fn, *, lr, epochs, batch_size, agg, threshold,
                min_labels=0, weight_decay=0.0, grad_clip=1.0, lr_lambda=None, save_path="best_trainable.pt",
                max_epoch_minutes=None, micro_batch=None, last_path=None, resume_from=None):
    """max_epoch_minutes: stop each epoch's training after this many minutes (batches are shuffled,
    so the epoch then covers a random part of the training windows). Dev scoring still runs.
    micro_batch: memory saver (gradient accumulation). Each batch of batch_size windows is run
    through the model micro_batch windows at a time and the gradients are summed before ONE
    optimizer step, so the update is the same as with the full batch. None = whole batch at once.
    last_path: after EVERY epoch, save a full checkpoint there (trainable weights of that epoch,
    the best epoch's weights, optimizer / scaler / schedule state, epoch number, best dev F1, log).
    resume_from: path to such a checkpoint (e.g. from an earlier notebook version attached as
    input). Training then continues at the next epoch and runs UP TO `epochs` in total, so
    epochs=6 after a 3-epoch run trains epochs 4, 5 and 6. The best weights so far are written
    to save_path first, so they are kept even if no new epoch beats them."""
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()
    t0, best_f1, best_res, log = time.time(), -1.0, None, []
    skipped = 0   # batches whose loss was NaN/inf (skipped, not trained on)
    start, best_state = 0, None
    if resume_from:
        ck = torch.load(resume_from, map_location="cpu", weights_only=False)
        r = model.load_state_dict(ck["trainable"], strict=False)
        names = {n for n, p in model.named_parameters() if p.requires_grad}
        assert not r.unexpected_keys and names <= set(ck["trainable"]), \
            f"checkpoint does not match this model (same LORMA_RANK / LORMA_TARGETS / head?) " \
            f"unexpected {r.unexpected_keys[:5]} missing {sorted(names - set(ck['trainable']))[:5]}"
        opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        if sched and ck.get("sched"): sched.load_state_dict(ck["sched"])
        start, best_f1, best_res, log = ck["epoch"], ck["best_f1"], ck["best_res"], ck["log"]
        skipped, best_state = ck.get("skipped", 0), ck["best"]
        torch.save(best_state, save_path)
        print(f"resumed from {resume_from}: {start} epochs done, best dev macro-F1 so far {best_f1:.4f}; "
              f"training epochs {start + 1}..{epochs}")
        assert start < epochs, f"checkpoint already has {start} epochs: set EPOCHS higher to train more"
    for epoch in range(start, epochs):
        model.train(); tot, nb = 0.0, 0
        batches = make_batches(train_ex, batch_size, shuffle=True)
        pbar = tqdm(batches, desc=f"epoch {epoch + 1}/{epochs}")
        t_ep, seen = time.time(), 0
        for batch in pbar:
            if max_epoch_minutes and time.time() - t_ep > 60 * max_epoch_minutes:
                tqdm.write(f"epoch {epoch + 1}: time cap {max_epoch_minutes} min reached after "
                           f"{seen}/{len(batches)} batches ({100 * seen / len(batches):.1f}% of the training windows)")
                break
            seen += 1
            opt.zero_grad(set_to_none=True)
            mb = micro_batch or len(batch)
            loss, bad = 0.0, False
            for i in range(0, len(batch), mb):
                part = batch[i:i + mb]
                b = to_tensors(part)
                with torch.autocast("cuda", dtype=torch.float16):
                    logits = model(b["ids"])
                l = loss_fn(logits.float(), b) * (len(part) / len(batch))   # weighted like the full-batch mean
                if not torch.isfinite(l):
                    bad = True
                    break
                scaler.scale(l).backward()
                loss += l.detach()
            if bad:
                opt.zero_grad(set_to_none=True)
                skipped += 1
                continue
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if sched: sched.step()
            tot += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{tot / nb:.4f}", skipped=skipped)
        res = evaluate(predict(model, val_ex, batch_size, agg, threshold, min_labels), val_docs, show=False)
        log.append({"epoch": epoch + 1, "train_loss": tot / max(1, nb), "batches": seen, "of": len(batches),
                    "dev_macro_f1": res["f1"], "dev_micro_f1": res["micro_f1"],
                    "dev_subset_acc": res["subset_acc"], "skipped": skipped})
        print(log[-1])
        cur = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
        if res["f1"] > best_f1:
            best_f1, best_res, best_state = res["f1"], {k: v for k, v in res.items() if k != "per"}, cur
            torch.save(cur, save_path)
        if last_path:
            torch.save({"trainable": cur, "best": best_state, "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "sched": sched.state_dict() if sched else None, "epoch": epoch + 1, "best_f1": best_f1,
                        "best_res": best_res, "log": log, "skipped": skipped}, last_path)
            tqdm.write(f"epoch {epoch + 1} checkpoint -> {last_path}")
    print(f"best dev macro-F1 {best_f1:.4f} (micro-F1 {best_res['micro_f1']:.4f}, subset acc {best_res['subset_acc']:.4f}) | "
          f"train time {(time.time() - t0) / 60:.1f} min | peak GPU mem {torch.cuda.max_memory_allocated() / 2**30:.2f} GB")
    return log, best_res


@torch.no_grad()
def time_inference(model, examples, batch_size, agg):
    """Wall-clock seconds to predict a set (for the efficiency table in the report)."""
    torch.cuda.synchronize(); t = time.time()
    predict_probs(model, examples, batch_size, agg)
    torch.cuda.synchronize(); return time.time() - t


def error_cases(pred, docs, k=5, chars=300):
    """Statutes most often missed / wrongly added on dev, and a few cases with both kinds of error."""
    missed, added = Counter(), Counter()
    for i, d in docs.items():
        g, p = set(d["labels"]), set(pred[i])
        missed.update(g - p); added.update(p - g)
    print("most often MISSED (statute: times):", missed.most_common(15))
    print("most often WRONGLY ADDED (statute: times):", added.most_common(15))
    bad = sorted(docs, key=lambda i: -len(set(docs[i]["labels"]) ^ set(pred[i])))[:k]
    for i in bad:
        g, p = set(docs[i]["labels"]), set(pred[i])
        t = docs[i]["text"]
        print(f"- {i} | {len(docs[i]['ids'])} tokens | gold {sorted(g)} | pred {sorted(p)} | "
              f"missed {sorted(g - p)} added {sorted(p - g)}\n  start: {t[:chars]!r}")


def write_submission(model, build_kwargs, batch_size, agg, threshold, min_labels=0, best_path=None,
                     out_dir="/kaggle/working"):
    """Reload the best trainable weights, predict test, write prediction.json + submission.zip (Cell D writer)."""
    if best_path:
        model.load_state_dict(torch.load(best_path), strict=False)
    test_ex = build_examples(test, **build_kwargs)
    write_prediction(predict(model, test_ex, batch_size, agg, threshold, min_labels), test, out_dir=out_dir)


# -------------------------------------------------------------------------------------
# Check + timing of FAST_ATTENTION on one batch of 4 windows x 1024 tokens (forward + backward)
# -------------------------------------------------------------------------------------
def _time_encoder(fast, reps=3):
    global FAST_ATTENTION
    FAST_ATTENTION = fast
    idx = torch.randint(4, cfg.vocab_size, (4, min(1024, cfg.max_seq_len)), device=device)   # small batch: timing only
    emb = lm.core.tok_emb.weight
    was = emb.requires_grad
    emb.requires_grad_(True)                    # makes sure backward has a path even if lm is frozen
    torch.cuda.synchronize(); t = time.time()
    for _ in range(reps):
        with torch.autocast("cuda", dtype=torch.float16):
            encoder_forward(lm, idx).float().mean().backward()
    torch.cuda.synchronize()
    emb.requires_grad_(was)
    for p in lm.parameters():
        p.grad = None                           # leave no gradients behind
    return (time.time() - t) / reps

lm.eval()
with torch.no_grad():
    _p = identity_probe()
    FAST_ATTENTION = True;  _f = encoder_forward(lm, _p)
    FAST_ATTENTION = False; _s = encoder_forward(lm, _p)
_diff = (_f - _s).abs().max().item()
try:
    _t_slow, _t_fast = _time_encoder(False), _time_encoder(True)
except torch.cuda.OutOfMemoryError:            # GPU still holds an earlier run: judge by equality only
    for p in lm.parameters():
        p.grad = None
    torch.cuda.empty_cache()
    _t_slow, _t_fast = float("nan"), float("nan")
    print("timing skipped (GPU memory busy): free the previous run's memory, see the thread")
FAST_ATTENTION = _diff < 1e-3 and not (_t_fast >= _t_slow)
print(f"fused attention: max diff {_diff:.2e} | step time plain {_t_slow:.3f}s vs fused {_t_fast:.3f}s "
      f"-> FAST_ATTENTION = {FAST_ATTENTION}")
torch.cuda.empty_cache()


fused attention: max diff 9.54e-07 | step time plain 0.284s vs fused 0.189s -> FAST_ATTENTION = True


## Cell E2 — LORMA (YOURS: fill the TODOs)

In [9]:
# =====================================================================================
# CELL E2 — LORMA on the c-6 simple encoder-decoder for LSI  <<< YOUR CELL >>>
# Filled with Rudra's own values (23 Oct), LORMALinear = Rudra's Phase 3 (I + BA)W layer, head = Rudra's mean+max head.
#
# Run after Cells 0, A, B, C, D and E0. To run again from scratch, re-run Cell A and E0 first
# (wrapping replaces the encoder's linears in place, and wrap_linears refuses to wrap twice).
#
# Already done for you in E0 (ancillary): the decoder is dropped, so for classification only
# the ENCODER runs: tokens -> 8 EncoderBlocks -> encoder_ln -> (B, T, 560) -> your head.
# encoder_linears / wrap_linears swap the chosen nn.Linear in every encoder block.
#
# YOURS (marked TODO):  [H] every value   [L] LORMALinear   [N] the head
# Sections marked [ANCILLARY] are plumbing; nothing to change there.
# =====================================================================================

# -------------------------------------------------------------------------------------
# [H] Hyperparameters and design choices — yours
# -------------------------------------------------------------------------------------
WINDOW        = 512   # tokens per window (<= 1536)
STRIDE        = 256   # step between window starts (<= WINDOW)
DOC_REPR      = "spread"   # long-document representation: "all", "first", "last", "first_last" or "spread"
MAX_WINDOWS   = 8   # only needed when DOC_REPR != "all"

LORMA_RANK    = 8
LORMA_TARGETS = ("qkv_proj", "out_proj", "ffn_in", "ffn_out")   # a tuple, any of "qkv_proj", "out_proj", "ffn_in", "ffn_out" (sizes printed below)
LR            = 3e-4
EPOCHS        = 6
MAX_EPOCH_MINUTES = None   # optional time cap per epoch (None = no cap)
BATCH_SIZE    = 16
WEIGHT_DECAY  = 0.01
LR_LAMBDA     = None   # optional: step -> LR multiplier, None = constant LR

# [ANCILLARY] GPU memory savers: same maths and same updates, only slower. Use them when a
# batch of long windows runs out of memory (CUDA OutOfMemoryError).
GRAD_CHECKPOINT = False    # True = recompute each encoder block in backward instead of storing it
MICRO_BATCH     = 4        # e.g. 4 = run each batch 4 windows at a time, one optimizer step per batch

# [ANCILLARY] continue an earlier run: path to its last_checkpoint.pt (attach that notebook's
# output as input; find the path with  !find /kaggle/input -name last_checkpoint.pt ).
# EPOCHS is the TOTAL: after a 3-epoch run, EPOCHS = 6 trains epochs 4-6. None = fresh start.
RESUME_FROM     = None

POS_WEIGHT_SCHEME = "sqrt_neg_pos"   # "none", "neg_pos" or "sqrt_neg_pos" (see make_pos_weight in E0)
POS_WEIGHT_CAP    = 10.0   # optional upper limit on any statute's weight

AGG           = "max"   # window probabilities -> case: "mean" or "max"
THRESHOLD     = 0.5   # fixed threshold used during training to pick the best epoch
MIN_LABELS    = 0   # at least this many statutes per case (0 = empty set allowed)
THRESH_MODE   = "global"   # after training, tuned on dev: "global", "per_label", or "fixed" (keep THRESHOLD)
THRESH_GRID   = [0.10, 0.15, 0.20, 0.25, 0.30,
                 0.35, 0.40, 0.45, 0.50, 0.55,
                 0.60, 0.65, 0.70, 0.75, 0.80]

SEED          = 42
for _n in ["WINDOW", "STRIDE", "DOC_REPR", "LORMA_RANK", "LORMA_TARGETS", "LR", "EPOCHS", "BATCH_SIZE",
           "WEIGHT_DECAY", "POS_WEIGHT_SCHEME", "AGG", "THRESHOLD", "MIN_LABELS", "THRESH_MODE"]:
    assert globals()[_n] is not None, f"set {_n}"
assert STRIDE <= WINDOW <= cfg.max_seq_len
assert DOC_REPR == "all" or MAX_WINDOWS, "set MAX_WINDOWS"
assert THRESH_MODE == "fixed" or THRESH_GRID, "set THRESH_GRID"


# -------------------------------------------------------------------------------------
# [ANCILLARY] run setup
# -------------------------------------------------------------------------------------
BUILD = dict(window=WINDOW, stride=STRIDE, doc_repr=DOC_REPR, max_windows=MAX_WINDOWS)
random.seed(SEED); torch.manual_seed(SEED)
RUN_NAME = f"c6_lorma_lsi_{DOC_REPR}"
OUT_DIR = f"/kaggle/working/{RUN_NAME}"; os.makedirs(OUT_DIR, exist_ok=True)
SAVE = f"{OUT_DIR}/best_trainable.pt"
LAST = f"{OUT_DIR}/last_checkpoint.pt"
POS_WEIGHT = make_pos_weight(POS_WEIGHT_SCHEME, POS_WEIGHT_CAP)

# Which linears can get LORMA, sizes from the loaded model (block 0; all 8 blocks are the same).
# Attention linears have no bias; the FFN ones do. qkv_proj is ONE fused Q,K,V projection.
for _name in ("qkv_proj", "out_proj", "ffn_in", "ffn_out"):
    _parent, _attr, _lin = next(encoder_linears(lm, (_name,)))
    print(f"  {_name:10s} {_lin.in_features:5d} -> {_lin.out_features:5d}  bias={_lin.bias is not None}")


# -------------------------------------------------------------------------------------
# [L] LORMA layer — yours (Phase 3): W' = (I + BA) W
# -------------------------------------------------------------------------------------
class LORMALinear(nn.Module):
    """Wraps ONE pretrained nn.Linear of the c-6 encoder.

    Constructed as   LORMALinear(base, rank)   by wrap_linears below, where
        base : the original nn.Linear (weight shape (out_features, in_features), bias may be None)
        rank : LORMA_RANK

    forward(x)
        x      : (B, T, in_features)    B windows, T tokens each (T = window length)
        return : (B, T, out_features)   must have the SAME shape as base(x), because it goes
                                        straight back into the encoder block

    What the plumbing expects:
        - the pretrained weights stay frozen (requires_grad False); only your new parameters
          train. Everything with requires_grad True is optimised and saved to best_trainable.pt.
        - keep self.base as an attribute.
        - if your initialisation makes the wrapped layer start out equal to base, the identity
          check below passes (diff ~0). If your design does not, set CHECK_IDENTITY = False.
    """
    def __init__(self, base: nn.Linear, rank: int):
        super().__init__()
        self.base = base
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = rank
        # Freeze pretrained weights
        for p in self.base.parameters():
            p.requires_grad = False

        # A: (rank, out_features), since A operates on Wx
        # B: (out_features, rank)
        self.lorma_A = nn.Parameter(
            torch.empty(self.rank, self.out_features)
        )
        self.lorma_B = nn.Parameter(
            torch.empty(self.out_features, self.rank)
        )

        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B)

    def forward(self, x):
        # Compute pretrained linear output without bias: Wx
        wx = F.linear(x, self.base.weight, bias=None)

        # Low-rank transformation of Wx: BA(Wx)
        lorma_out = F.linear(
            F.linear(wx, self.lorma_A),
            self.lorma_B
        )

        # Return Wx + BA(Wx) + b
        if self.base.bias is not None:
            return wx + lorma_out + self.base.bias

        return wx + lorma_out


# [ANCILLARY] quick shape test on a dummy linear (no model involved): catches shape bugs before wrapping
_t = LORMALinear(nn.Linear(560, 1680, bias=False), rank=LORMA_RANK)
_y = _t(torch.randn(2, 7, 560))
assert _y.shape == (2, 7, 1680), f"LORMALinear output shape {tuple(_y.shape)}, expected (2, 7, 1680)"
print("LORMALinear trainable params on a 560->1680 layer:", sum(p.numel() for p in _t.parameters() if p.requires_grad))
del _t, _y


# -------------------------------------------------------------------------------------
# [ANCILLARY] freeze c-6, wrap the chosen linears, check identity at init
# -------------------------------------------------------------------------------------
CHECK_IDENTITY = True

lm.eval()
probe = identity_probe()
with torch.no_grad():
    ref = features(probe).float()

for p in lm.parameters():
    p.requires_grad = False
n_wrapped = wrap_linears(lm, LORMA_TARGETS, lambda lin: LORMALinear(lin, LORMA_RANK).to(device))
print(f"wrapped {n_wrapped} linears ({len(LORMA_TARGETS)} per block x {cfg.n_layers} blocks)")

if CHECK_IDENTITY:
    with torch.no_grad():
        diff = (features(probe).float() - ref).abs().max().item()
    print(f"LORMA identity-at-init max |diff| = {diff:.2e} (should be ~0)")
    assert diff < 1e-3, "wrapped model differs from the original at init"


# -------------------------------------------------------------------------------------
# [N] Head — yours (mean + max pool) (multi-label: one logit per statute, no softmax, BCE is applied in the loss)
#     input  h      : (B, T, 560)        encoder vectors of B windows (never padded, every token is real)
#     output logits : (B, n_labels)      n_labels = N_LABELS statutes (column j = statute STATUTES[j])
# -------------------------------------------------------------------------------------
class LSIHead(nn.Module):
    def __init__(self, in_dim, n_labels):
        super().__init__()
        self.norm = nn.LayerNorm(2 * in_dim)
        self.mlp = nn.Sequential(
            nn.Linear(2 * in_dim, in_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(in_dim, n_labels)
        )

    def forward(self, h):
        mean_pool = h.mean(dim=1)
        max_pool = h.max(dim=1).values
        pooled = torch.cat([mean_pool, max_pool], dim=-1)
        return self.mlp(self.norm(pooled))

head = LSIHead(FEAT_DIM, N_LABELS)
model = LSIModel(lm, head).to(device)
count_params(model)
with torch.no_grad():
    assert model(probe).shape == (1, N_LABELS), "head must return (B, N_LABELS)"


# -------------------------------------------------------------------------------------
# [ANCILLARY] data, training (best epoch by dev macro-F1 at THRESHOLD), threshold tuning, scoring
# -------------------------------------------------------------------------------------
train_ex = build_examples(train, **BUILD)
val_ex = build_examples(dev, **BUILD)

log, best_res = train_model(
    model, train_ex, val_ex, dev,
    loss_fn=lambda logits, b: bce_loss(logits, b["labels"], POS_WEIGHT),
    lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
    lr_lambda=LR_LAMBDA, save_path=SAVE, agg=AGG, threshold=THRESHOLD, min_labels=MIN_LABELS,
    max_epoch_minutes=MAX_EPOCH_MINUTES, micro_batch=MICRO_BATCH,
    last_path=LAST, resume_from=RESUME_FROM)
train_mem = torch.cuda.max_memory_allocated() / 2**30

model.load_state_dict(torch.load(SAVE), strict=False)
dev_probs = predict_probs(model, val_ex, BATCH_SIZE, AGG)
print(f"\n-- dev at the fixed THRESHOLD {THRESHOLD}")
evaluate(decide(dev_probs, THRESHOLD, MIN_LABELS), dev)
FINAL_THR = THRESHOLD if THRESH_MODE == "fixed" else tune_thresholds(dev_probs, dev, THRESH_MODE, THRESH_GRID, MIN_LABELS)
dev_pred = decide(dev_probs, FINAL_THR, MIN_LABELS)
print(f"\n-- dev with the final thresholds ({THRESH_MODE})")
res = evaluate(dev_pred, dev)
per_statute_report(res)
error_cases(dev_pred, dev)
print(f"dev inference time {time_inference(model, val_ex, BATCH_SIZE, AGG):.1f} s | peak training memory {train_mem:.2f} GB")
json.dump({"run": RUN_NAME, "build": BUILD, "log": log, "thresh_mode": THRESH_MODE,
           "final_thr": np.asarray(FINAL_THR).tolist(),
           "dev": {k: res[k] for k in ("f1", "f1_all", "micro_f1", "subset_acc")},
           "dev_probs": {k: v.round(4).tolist() for k, v in dev_probs.items()}},
          open(f"{OUT_DIR}/dev_log.json", "w"))

# Predict test with the best weights and final thresholds -> /kaggle/working/<RUN_NAME>/submission.zip
write_submission(model, BUILD, BATCH_SIZE, AGG, FINAL_THR, MIN_LABELS, best_path=SAVE, out_dir=OUT_DIR)

# [ANCILLARY] everything a KD notebook needs to rebuild this model as the teacher
json.dump({"run": RUN_NAME, "build": BUILD, "lorma_rank": LORMA_RANK, "lorma_targets": list(LORMA_TARGETS),
           "agg": AGG, "min_labels": MIN_LABELS, "final_thr": np.asarray(FINAL_THR).tolist(),
           "statutes": [int(s) for s in STATUTES], "weights": "best_trainable.pt"},
          open(f"{OUT_DIR}/teacher_config.json", "w"), indent=1)
print("files for later runs / KD:", sorted(os.listdir(OUT_DIR)))


pos_weight: min 1.63 median 7.84 max 10.00
  qkv_proj     560 ->  1680  bias=False
  out_proj     560 ->   560  bias=False
  ffn_in       560 ->  1000  bias=True
  ffn_out     1000 ->   560  bias=True
LORMALinear trainable params on a 560->1680 layer: 26880
wrapped 32 linears (4 per block x 8 blocks)
LORMA identity-at-init max |diff| = 9.54e-07 (should be ~0)
trainable 1.173M / total 27.23M (4.31%)


windowing:   0%|          | 0/42750 [00:00<?, ?it/s]

42750 cases -> 161129 windows | windows per case: mean 3.77 max 8


windowing:   0%|          | 0/10181 [00:00<?, ?it/s]

10181 cases -> 41971 windows | windows per case: mean 4.12 max 8


epoch 1/6:   0%|          | 0/10306 [00:00<?, ?it/s]

... (21 outputs left out)

windowing:   0%|          | 0/13019 [00:00<?, ?it/s]

13019 cases -> 53109 windows | windows per case: mean 4.08 max 8


predict:   0%|          | 0/3557 [00:00<?, ?it/s]

WARNING 603 test cases have an empty pred list (the spec does not say if [] is accepted)
13019 cases | statutes per case mean 6.86 max 38 | distinct statutes predicted 100 -> /kaggle/working/c6_lorma_lsi_spread/submission.zip
files for later runs / KD: ['best_trainable.pt', 'dev_log.json', 'last_checkpoint.pt', 'prediction.json', 'submission.zip', 'teacher_config.json']


In [10]:
# # =====================================================================================
# # CELL E2 — LORMA on the c-6 simple encoder-decoder for LSI  <<< YOUR CELL >>>
# #
# # Run after Cells 0, A, B, C, D and E0. To run again from scratch, re-run Cell A and E0 first
# # (wrapping replaces the encoder's linears in place, and wrap_linears refuses to wrap twice).
# #
# # Already done for you in E0 (ancillary): the decoder is dropped, so for classification only
# # the ENCODER runs: tokens -> 8 EncoderBlocks -> encoder_ln -> (B, T, 560) -> your head.
# # encoder_linears / wrap_linears swap the chosen nn.Linear in every encoder block.
# #
# # YOURS (marked TODO):  [H] every value   [L] LORMALinear   [N] the head
# # Sections marked [ANCILLARY] are plumbing; nothing to change there.
# # =====================================================================================

# # -------------------------------------------------------------------------------------
# # [H] Hyperparameters and design choices — TODO: all yours. None = not set; the assert stops you.
# # -------------------------------------------------------------------------------------
# WINDOW        = None   # tokens per window (<= 1536)
# STRIDE        = None   # step between window starts (<= WINDOW)
# DOC_REPR      = None   # long-document representation: "all", "first", "last", "first_last" or "spread"
# MAX_WINDOWS   = None   # only needed when DOC_REPR != "all"

# LORMA_RANK    = None
# LORMA_TARGETS = None   # a tuple, any of "qkv_proj", "out_proj", "ffn_in", "ffn_out" (sizes printed below)
# LR            = None
# EPOCHS        = None
# MAX_EPOCH_MINUTES = None   # optional time cap per epoch (None = no cap)
# BATCH_SIZE    = None
# WEIGHT_DECAY  = None
# LR_LAMBDA     = None   # optional: step -> LR multiplier, None = constant LR

# # [ANCILLARY] GPU memory savers: same maths and same updates, only slower. Use them when a
# # batch of long windows runs out of memory (CUDA OutOfMemoryError).
# GRAD_CHECKPOINT = False    # True = recompute each encoder block in backward instead of storing it
# MICRO_BATCH     = None     # e.g. 4 = run each batch 4 windows at a time, one optimizer step per batch

# # [ANCILLARY] continue an earlier run: path to its last_checkpoint.pt (attach that notebook's
# # output as input; find the path with  !find /kaggle/input -name last_checkpoint.pt ).
# # EPOCHS is the TOTAL: after a 3-epoch run, EPOCHS = 6 trains epochs 4-6. None = fresh start.
# RESUME_FROM     = None

# POS_WEIGHT_SCHEME = None   # "none", "neg_pos" or "sqrt_neg_pos" (see make_pos_weight in E0)
# POS_WEIGHT_CAP    = None   # optional upper limit on any statute's weight

# AGG           = None   # window probabilities -> case: "mean" or "max"
# THRESHOLD     = None   # fixed threshold used during training to pick the best epoch
# MIN_LABELS    = None   # at least this many statutes per case (0 = empty set allowed)
# THRESH_MODE   = None   # after training, tuned on dev: "global", "per_label", or "fixed" (keep THRESHOLD)
# THRESH_GRID   = None   # candidate thresholds for tuning, e.g. a list of floats (not needed for "fixed")

# SEED          = 42
# for _n in ["WINDOW", "STRIDE", "DOC_REPR", "LORMA_RANK", "LORMA_TARGETS", "LR", "EPOCHS", "BATCH_SIZE",
#            "WEIGHT_DECAY", "POS_WEIGHT_SCHEME", "AGG", "THRESHOLD", "MIN_LABELS", "THRESH_MODE"]:
#     assert globals()[_n] is not None, f"set {_n}"
# assert STRIDE <= WINDOW <= cfg.max_seq_len
# assert DOC_REPR == "all" or MAX_WINDOWS, "set MAX_WINDOWS"
# assert THRESH_MODE == "fixed" or THRESH_GRID, "set THRESH_GRID"


# # -------------------------------------------------------------------------------------
# # [ANCILLARY] run setup
# # -------------------------------------------------------------------------------------
# BUILD = dict(window=WINDOW, stride=STRIDE, doc_repr=DOC_REPR, max_windows=MAX_WINDOWS)
# random.seed(SEED); torch.manual_seed(SEED)
# RUN_NAME = f"c6_lorma_lsi_{DOC_REPR}"
# OUT_DIR = f"/kaggle/working/{RUN_NAME}"; os.makedirs(OUT_DIR, exist_ok=True)
# SAVE = f"{OUT_DIR}/best_trainable.pt"
# LAST = f"{OUT_DIR}/last_checkpoint.pt"
# POS_WEIGHT = make_pos_weight(POS_WEIGHT_SCHEME, POS_WEIGHT_CAP)

# # Which linears can get LORMA, sizes from the loaded model (block 0; all 8 blocks are the same).
# # Attention linears have no bias; the FFN ones do. qkv_proj is ONE fused Q,K,V projection.
# for _name in ("qkv_proj", "out_proj", "ffn_in", "ffn_out"):
#     _parent, _attr, _lin = next(encoder_linears(lm, (_name,)))
#     print(f"  {_name:10s} {_lin.in_features:5d} -> {_lin.out_features:5d}  bias={_lin.bias is not None}")


# # -------------------------------------------------------------------------------------
# # [L] LORMA layer — TODO: yours
# # -------------------------------------------------------------------------------------
# class LORMALinear(nn.Module):
#     """Wraps ONE pretrained nn.Linear of the c-6 encoder.

#     Constructed as   LORMALinear(base, rank)   by wrap_linears below, where
#         base : the original nn.Linear (weight shape (out_features, in_features), bias may be None)
#         rank : LORMA_RANK

#     forward(x)
#         x      : (B, T, in_features)    B windows, T tokens each (T = window length)
#         return : (B, T, out_features)   must have the SAME shape as base(x), because it goes
#                                         straight back into the encoder block

#     What the plumbing expects:
#         - the pretrained weights stay frozen (requires_grad False); only your new parameters
#           train. Everything with requires_grad True is optimised and saved to best_trainable.pt.
#         - keep self.base as an attribute.
#         - if your initialisation makes the wrapped layer start out equal to base, the identity
#           check below passes (diff ~0). If your design does not, set CHECK_IDENTITY = False.
#     """
#     def __init__(self, base: nn.Linear, rank: int):
#         super().__init__()
#         self.base = base
#         self.in_features = base.in_features
#         self.out_features = base.out_features
#         self.rank = rank
#         # TODO: freeze base, create and initialise your LORMA parameters
#         raise NotImplementedError("write LORMALinear.__init__")

#     def forward(self, x):
#         # TODO: x (B, T, in_features) -> (B, T, out_features)
#         raise NotImplementedError("write LORMALinear.forward")


# # [ANCILLARY] quick shape test on a dummy linear (no model involved): catches shape bugs before wrapping
# _t = LORMALinear(nn.Linear(560, 1680, bias=False), rank=LORMA_RANK)
# _y = _t(torch.randn(2, 7, 560))
# assert _y.shape == (2, 7, 1680), f"LORMALinear output shape {tuple(_y.shape)}, expected (2, 7, 1680)"
# print("LORMALinear trainable params on a 560->1680 layer:", sum(p.numel() for p in _t.parameters() if p.requires_grad))
# del _t, _y


# # -------------------------------------------------------------------------------------
# # [ANCILLARY] freeze c-6, wrap the chosen linears, check identity at init
# # -------------------------------------------------------------------------------------
# CHECK_IDENTITY = True

# lm.eval()
# probe = identity_probe()
# with torch.no_grad():
#     ref = features(probe).float()

# for p in lm.parameters():
#     p.requires_grad = False
# n_wrapped = wrap_linears(lm, LORMA_TARGETS, lambda lin: LORMALinear(lin, LORMA_RANK).to(device))
# print(f"wrapped {n_wrapped} linears ({len(LORMA_TARGETS)} per block x {cfg.n_layers} blocks)")

# if CHECK_IDENTITY:
#     with torch.no_grad():
#         diff = (features(probe).float() - ref).abs().max().item()
#     print(f"LORMA identity-at-init max |diff| = {diff:.2e} (should be ~0)")
#     assert diff < 1e-3, "wrapped model differs from the original at init"


# # -------------------------------------------------------------------------------------
# # [N] Head — TODO: yours (multi-label: one logit per statute, no softmax, BCE is applied in the loss)
# #     input  h      : (B, T, 560)        encoder vectors of B windows (never padded, every token is real)
# #     output logits : (B, n_labels)      n_labels = N_LABELS statutes (column j = statute STATUTES[j])
# # -------------------------------------------------------------------------------------
# class LSIHead(nn.Module):
#     def __init__(self, in_dim, n_labels):
#         super().__init__()
#         # TODO
#         raise NotImplementedError("write LSIHead")

#     def forward(self, h):
#         # TODO: (B, T, in_dim) -> (B, n_labels)
#         raise NotImplementedError

# head = LSIHead(FEAT_DIM, N_LABELS)
# model = LSIModel(lm, head).to(device)
# count_params(model)
# with torch.no_grad():
#     assert model(probe).shape == (1, N_LABELS), "head must return (B, N_LABELS)"


# # -------------------------------------------------------------------------------------
# # [ANCILLARY] data, training (best epoch by dev macro-F1 at THRESHOLD), threshold tuning, scoring
# # -------------------------------------------------------------------------------------
# train_ex = build_examples(train, **BUILD)
# val_ex = build_examples(dev, **BUILD)

# log, best_res = train_model(
#     model, train_ex, val_ex, dev,
#     loss_fn=lambda logits, b: bce_loss(logits, b["labels"], POS_WEIGHT),
#     lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
#     lr_lambda=LR_LAMBDA, save_path=SAVE, agg=AGG, threshold=THRESHOLD, min_labels=MIN_LABELS,
#     max_epoch_minutes=MAX_EPOCH_MINUTES, micro_batch=MICRO_BATCH,
#     last_path=LAST, resume_from=RESUME_FROM)
# train_mem = torch.cuda.max_memory_allocated() / 2**30

# model.load_state_dict(torch.load(SAVE), strict=False)
# dev_probs = predict_probs(model, val_ex, BATCH_SIZE, AGG)
# print(f"\n-- dev at the fixed THRESHOLD {THRESHOLD}")
# evaluate(decide(dev_probs, THRESHOLD, MIN_LABELS), dev)
# FINAL_THR = THRESHOLD if THRESH_MODE == "fixed" else tune_thresholds(dev_probs, dev, THRESH_MODE, THRESH_GRID, MIN_LABELS)
# dev_pred = decide(dev_probs, FINAL_THR, MIN_LABELS)
# print(f"\n-- dev with the final thresholds ({THRESH_MODE})")
# res = evaluate(dev_pred, dev)
# per_statute_report(res)
# error_cases(dev_pred, dev)
# print(f"dev inference time {time_inference(model, val_ex, BATCH_SIZE, AGG):.1f} s | peak training memory {train_mem:.2f} GB")
# json.dump({"run": RUN_NAME, "build": BUILD, "log": log, "thresh_mode": THRESH_MODE,
#            "final_thr": np.asarray(FINAL_THR).tolist(),
#            "dev": {k: res[k] for k in ("f1", "f1_all", "micro_f1", "subset_acc")},
#            "dev_probs": {k: v.round(4).tolist() for k, v in dev_probs.items()}},
#           open(f"{OUT_DIR}/dev_log.json", "w"))

# # Predict test with the best weights and final thresholds -> /kaggle/working/<RUN_NAME>/submission.zip
# write_submission(model, BUILD, BATCH_SIZE, AGG, FINAL_THR, MIN_LABELS, best_path=SAVE, out_dir=OUT_DIR)

# # [ANCILLARY] everything a KD notebook needs to rebuild this model as the teacher
# json.dump({"run": RUN_NAME, "build": BUILD, "lorma_rank": LORMA_RANK, "lorma_targets": list(LORMA_TARGETS),
#            "agg": AGG, "min_labels": MIN_LABELS, "final_thr": np.asarray(FINAL_THR).tolist(),
#            "statutes": [int(s) for s in STATUTES], "weights": "best_trainable.pt"},
#           open(f"{OUT_DIR}/teacher_config.json", "w"), indent=1)
# print("files for later runs / KD:", sorted(os.listdir(OUT_DIR)))


## Cell E3a — [ANCILLARY] save teacher probabilities for KD (optional)

In [11]:
# =====================================================================================
# CELL E3a — [ANCILLARY] save the LORMA model's predictions as KD teacher signals (optional)
# Run right after Cell E2 in the same session (uses model, SAVE, train_ex, val_ex, BUILD,
# BATCH_SIZE, AGG, FINAL_THR, OUT_DIR, STATUTES).
#
# Saves one probability per statute for every TRAIN and DEV case (case level, after AGG),
# plus test, to <OUT_DIR>/teacher_probs.json. Case-level probabilities over the statute list
# don't depend on the tokenizer or the window size, so a student on ANY Phase 1 backbone
# (c-1, c-3, c-5 or a smaller c-6) can learn from them in another notebook: add this
# notebook's output as an input dataset there and load the file.
# Forward passes only (no training): roughly the cost of one dev scoring per split.
# =====================================================================================
model.load_state_dict(torch.load(SAVE), strict=False)
model.eval()

teacher = {"run": RUN_NAME, "build": BUILD, "agg": AGG, "statutes": [int(s) for s in STATUTES],
           "final_thr": np.asarray(FINAL_THR).tolist()}
for name, ex in (("train", train_ex), ("dev", val_ex), ("test", build_examples(test, **BUILD))):
    probs = predict_probs(model, ex, BATCH_SIZE, AGG)
    teacher[name] = {k: v.round(5).tolist() for k, v in probs.items()}
    print(f"{name}: {len(probs)} cases")

path = f"{OUT_DIR}/teacher_probs.json"
json.dump(teacher, open(path, "w"))
print(f"saved -> {path} ({os.path.getsize(path) / 2**20:.1f} MB) | columns = statutes in this order: {STATUTES[:10]} ...")


windowing:   0%|          | 0/13019 [00:00<?, ?it/s]

13019 cases -> 53109 windows | windows per case: mean 4.08 max 8


predict:   0%|          | 0/10306 [00:00<?, ?it/s]

train: 42750 cases


predict:   0%|          | 0/2906 [00:00<?, ?it/s]

dev: 10181 cases


predict:   0%|          | 0/3557 [00:00<?, ?it/s]

test: 13019 cases
saved -> /kaggle/working/c6_lorma_lsi_spread/teacher_probs.json (127.7 MB) | columns = statutes in this order: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] ...
